In [5]:
from cobra import Model, Reaction, Metabolite
from cobra.flux_analysis import flux_variability_analysis
model = Model("toy_fba_model")
mets = {}
metabolite_names = [
    "Carbon1",
    "Carbon2",
    "A",
    "B",
    "C",
    "D",
    "E",
    "F",
    "G",
    "H",
    "I",
    "O2",
    "ATP",
    "NADH"
]
for met in metabolite_names:
    mets[met] = Metabolite(met)

def create_reaction(rxn_id, stoich, lb=0, ub=1000):
    rxn = Reaction(rxn_id)
    rxn.lower_bound = lb
    rxn.upper_bound = ub
    rxn.add_metabolites(stoich)
    return rxn

# INTERNAL REACTIONS (R1-R13)
R1 = create_reaction(
    "R1",
    {
        mets["Carbon1"]: -1,
        mets["A"]: 1
    }
)
R2 = create_reaction(
    "R2",
    {
        mets["Carbon2"]: -1,
        mets["A"]: 1
    }
)
R3 = create_reaction(
    "R3",
    {
        mets["A"]: -1,
        mets["ATP"]: -1,
        mets["B"]: 1
    }
)
R4 = create_reaction(
    "R4",
    {
        mets["A"]: -1,
        mets["I"]: 1
    }
)
R5 = create_reaction(
    "R5",
    {
        mets["I"]: -1,
        mets["ATP"]: -1,
        mets["B"]: 1
    }
)
R6 = create_reaction(
    "R6",
    {
        mets["B"]: -1,
        mets["C"]: 1,
        mets["ATP"]: 2,
        mets["NADH"]: 2
    }
)
R7 = create_reaction(
    "R7",
    {
        mets["B"]: -1,
        mets["F"]: 1
    }
)
R8 = create_reaction(
    "R8",
    {
        mets["C"]: -1,
        mets["G"]: 1
    }
)
R9 = create_reaction(
    "R9",
    {
        mets["G"]: -1,
        mets["C"]: 0.8,
        mets["NADH"]: 2
    },
    lb=-1000,
    ub=1000
)
R10 = create_reaction(
    "R10",
    {
        mets["C"]: -1,
        mets["D"]: 3,
        mets["ATP"]: 2
    }
)
R11 = create_reaction(
    "R11",
    {
        mets["C"]: -1,
        mets["NADH"]: -4,
        mets["E"]: 3
    }
)
R12 = create_reaction(
    "R12",
    {
        mets["G"]: -1,
        mets["ATP"]: -1,
        mets["NADH"]: -2,
        mets["H"]: 1
    },
    lb=-1000,
    ub=1000
)
R13 = create_reaction(
    "R13",
    {
        mets["O2"]: -1,
        mets["NADH"]: -1,
        mets["ATP"]: 1
    }
)

# EXCHANGE REACTIONS (E1-E7)
E1 = create_reaction(
    "E1",
    {
        mets["Carbon1"]: -1
    },
    lb=-10,
    ub=1000
)
E2 = create_reaction(
    "E2",
    {
        mets["Carbon2"]: -1
    },
    lb=0,
    ub=1000
)
E3 = create_reaction(
    "E3",
    {
        mets["O2"]: -1
    },
    lb=-1000,
    ub=1000
)
E4 = create_reaction(
    "E4",
    {
        mets["D"]: -1
    },
    lb=0,
    ub=1000
)
E5 = create_reaction(
    "E5",
    {
        mets["E"]: -1
    },
    lb=0,
    ub=1000
)
E6 = create_reaction(
    "E6",
    {
        mets["F"]: -1
    },
    lb=0,
    ub=1000
)
E7 = create_reaction(
    "E7",
    {
        mets["H"]: -1
    },
    lb=0,
    ub=1000
)

# BIOMASS REACTION
BIOMASS = create_reaction(
    "BIOMASS",
    {
        mets["C"]: -1,
        mets["F"]: -1,
        mets["H"]: -1,
        mets["ATP"]: -10
    }
)

model.add_reactions([R1, R2, R3, R4, R5,R6, R7, R8, R9, R10, R11, R12, R13, E1, E2, E3, E4, E5, E6, E7, BIOMASS])
model.objective = "BIOMASS"

In [16]:
#FBA
wt = model.optimize()
print("\n===================================")
print("OPTIMAL GROWTH RATE")
print("===================================\n")
print(wt.objective_value)
print("\n===================================")
print("FLUX DISTRIBUTION")
print("===================================\n")
print(wt.fluxes)


OPTIMAL GROWTH RATE

2.765957446808511

FLUX DISTRIBUTION

R1         10.000000
R2          0.000000
R3         10.000000
R4          0.000000
R5          0.000000
R6          7.234043
R7          2.765957
R8         11.276596
R9          8.510638
R10         0.000000
R11         0.000000
R12         2.765957
R13        25.957447
E1        -10.000000
E2          0.000000
E3        -25.957447
E4          0.000000
E5          0.000000
E6          0.000000
E7          0.000000
BIOMASS     2.765957
Name: fluxes, dtype: float64


In [7]:
#FVA
print("\n===================================")
print("FLUX VARIABILITY ANALYSIS")
print("===================================\n")
fva = flux_variability_analysis(
    model,
    fraction_of_optimum=1.0
)
print(fva)


FLUX VARIABILITY ANALYSIS

           minimum       maximum
R1       10.000000  1.000000e+01
R2        0.000000  0.000000e+00
R3        0.000000  1.000000e+01
R4        0.000000  1.000000e+01
R5        0.000000  1.000000e+01
R6        7.234043  7.234043e+00
R7        2.765957  2.765957e+00
R8       11.276596  1.127660e+01
R9        8.510638  8.510638e+00
R10       0.000000  4.433653e-15
R11       0.000000  2.533516e-15
R12       2.765957  2.765957e+00
R13      25.957447  2.595745e+01
E1      -10.000000 -1.000000e+01
E2        0.000000  0.000000e+00
E3      -25.957447 -2.595745e+01
E4        0.000000  1.330096e-14
E5        0.000000  7.600548e-15
E6        0.000000  2.533516e-15
E7        0.000000  2.728402e-15
BIOMASS   2.765957  2.765957e+00


In [8]:
# Example 2: anerobic condition
with model:
    model.reactions.E3.lower_bound = 0
    solution = model.optimize()
    print(
        "Anaerobic growth:",
        solution.objective_value
    )
    print(solution.fluxes)

Anaerobic growth: 1.2820512820512824
R1         10.000000
R2          0.000000
R3         10.000000
R4          0.000000
R5          0.000000
R6          8.717949
R7          1.282051
R8          0.000000
R9         -1.282051
R10         3.333333
R11         3.076923
R12         1.282051
R13         0.000000
E1        -10.000000
E2          0.000000
E3          0.000000
E4         10.000000
E5          9.230769
E6          0.000000
E7          0.000000
BIOMASS     1.282051
Name: fluxes, dtype: float64


In [22]:
#Example 3: Genetic perturbation under carbon-limited aerobic condition
with model:
    # AEROBIC CONDITION
    model.reactions.E3.lower_bound = -1000
    # KNOCK OUT R7
    model.reactions.R7.knock_out()
    solution = model.optimize()
    print("\n===================================")
    print("GENETIC PERTURBATION (R7 KO)")
    print("===================================\n")
    print(
        "Growth rate:",
        round(solution.objective_value,6)
    )
    print("\n===================================")
    print("FLUX DISTRIBUTION")
    print("===================================\n")
    print(
        solution.fluxes.round(6)
    )
comparison = (
    solution.fluxes
    -
    wt.fluxes
)
print("\n===================================")
print("FLUX CHANGES")
print("===================================\n")
print(
    comparison.round(6)
)


GENETIC PERTURBATION (R7 KO)

Growth rate: -0.0

FLUX DISTRIBUTION

R1         0.0
R2         0.0
R3         0.0
R4         0.0
R5         0.0
R6        -0.0
R7         0.0
R8        -0.0
R9         0.0
R10        0.0
R11        0.0
R12       -0.0
R13       -0.0
E1         0.0
E2         0.0
E3         0.0
E4         0.0
E5         0.0
E6         0.0
E7         0.0
BIOMASS   -0.0
Name: fluxes, dtype: float64

FLUX CHANGES

R1        -10.000000
R2          0.000000
R3        -10.000000
R4          0.000000
R5          0.000000
R6         -7.234043
R7         -2.765957
R8        -11.276596
R9         -8.510638
R10         0.000000
R11         0.000000
R12        -2.765957
R13       -25.957447
E1         10.000000
E2          0.000000
E3         25.957447
E4          0.000000
E5          0.000000
E6          0.000000
E7          0.000000
BIOMASS    -2.765957
Name: fluxes, dtype: float64


In [20]:
with model:
    # AEROBIC CONDITION
    model.reactions.E3.lower_bound = -1000
    # KNOCK OUT R8
    model.reactions.R8.knock_out()
    solution = model.optimize()
    print("\n===================================")
    print("GENETIC PERTURBATION (R8 KO)")
    print("===================================\n")
    print(
        "Growth rate:",
        round(solution.objective_value,6)
    )
    print("\n===================================")
    print("FLUX DISTRIBUTION")
    print("===================================\n")
    print(
        solution.fluxes.round(6)
    )
comparison = (
    solution.fluxes
    -
    wt.fluxes
)
print("\n===================================")
print("FLUX CHANGES")
print("===================================\n")
print(
    comparison.round(6)
)


GENETIC PERTURBATION (R8 KO)

Growth rate: 2.03252

FLUX DISTRIBUTION

R1         10.000000
R2          0.000000
R3         10.000000
R4          0.000000
R5          0.000000
R6          7.967480
R7          2.032520
R8          0.000000
R9         -2.032520
R10         4.308943
R11         0.000000
R12         2.032520
R13         7.804878
E1        -10.000000
E2          0.000000
E3         -7.804878
E4         12.926829
E5          0.000000
E6          0.000000
E7          0.000000
BIOMASS     2.032520
Name: fluxes, dtype: float64

FLUX CHANGES

R1          0.000000
R2          0.000000
R3          0.000000
R4          0.000000
R5          0.000000
R6          0.733437
R7         -0.733437
R8        -11.276596
R9        -10.543159
R10         4.308943
R11         0.000000
R12        -0.733437
R13       -18.152569
E1          0.000000
E2          0.000000
E3         18.152569
E4         12.926829
E5          0.000000
E6          0.000000
E7          0.000000
BIOMASS    -0.733437
Na

In [21]:
with model:
    # AEROBIC CONDITION
    model.reactions.E3.lower_bound = -1000
    # KNOCK OUT R9
    model.reactions.R9.knock_out()
    solution = model.optimize()
    print("\n===================================")
    print("GENETIC PERTURBATION (R9 KO)")
    print("===================================\n")
    print(
        "Growth rate:",
        round(solution.objective_value,6)
    )
    print("\n===================================")
    print("FLUX DISTRIBUTION")
    print("===================================\n")
    print(
        solution.fluxes.round(6)
    )
comparison = (
    solution.fluxes
    -
    wt.fluxes
)
print("\n===================================")
print("FLUX CHANGES")
print("===================================\n")
print(
    comparison.round(6)
)


GENETIC PERTURBATION (R9 KO)

Growth rate: 2.173913

FLUX DISTRIBUTION

R1         10.000000
R2          0.000000
R3         10.000000
R4          0.000000
R5          0.000000
R6          7.826087
R7          2.173913
R8          2.173913
R9          0.000000
R10         3.478261
R11         0.000000
R12         2.173913
R13        11.304348
E1        -10.000000
E2          0.000000
E3        -11.304348
E4         10.434783
E5          0.000000
E6          0.000000
E7          0.000000
BIOMASS     2.173913
Name: fluxes, dtype: float64

FLUX CHANGES

R1          0.000000
R2          0.000000
R3          0.000000
R4          0.000000
R5          0.000000
R6          0.592044
R7         -0.592044
R8         -9.102683
R9         -8.510638
R10         3.478261
R11         0.000000
R12        -0.592044
R13       -14.653099
E1          0.000000
E2          0.000000
E3         14.653099
E4         10.434783
E5          0.000000
E6          0.000000
E7          0.000000
BIOMASS    -0.592044
N

In [19]:
comparison = (
    solution.fluxes
    -
    wt.fluxes
)
print("\n===================================")
print("FLUX CHANGES")
print("===================================\n")
print(
    comparison.round(6)
)


FLUX CHANGES

R1          0.000000
R2          0.000000
R3          0.000000
R4          0.000000
R5          0.000000
R6          0.592044
R7         -0.592044
R8         -9.102683
R9         -8.510638
R10         3.478261
R11         0.000000
R12        -0.592044
R13       -14.653099
E1          0.000000
E2          0.000000
E3         14.653099
E4         10.434783
E5          0.000000
E6          0.000000
E7          0.000000
BIOMASS    -0.592044
Name: fluxes, dtype: float64
